# Notebook 01 — LAV-DF Setup, Schema Verification, and Manifest Generation

**Project:** AV-XAI (Explainable Multimodal Deepfake Detection)
**Dataset:** LAV-DF (Localized Audio-Visual DeepFake, Cai et al. CVIU 2023)
**Source:** https://huggingface.co/datasets/ControlNet/LAV-DF

## What this notebook does (smoke-test → main-run pattern)

1. **Phase A** — Verify Hugging Face authentication (the dataset is gated)
2. **Phase B** — Download metadata only (33 MB) — fast fail if anything is wrong
3. **Phase C** — Inspect the actual metadata schema and assert our assumptions
4. **Phase D** — Compute class distribution, actor counts, verify official splits are actor-disjoint
5. **Phase E** — Subsample to a balanced 30K target (within official train/dev/test splits)
6. **Phase F** — Write `manifest.csv` and `splits/{train,val,test}.csv`
7. **Phase G** — Selectively download only the ~30K MP4s in the manifest (~5–6 GB, not 25 GB)
8. **Phase H** — Final verification: file existence, actor leakage, class balance

## Prerequisites (do these BEFORE running)

1. Create a Hugging Face account if you don't have one.
2. Visit https://huggingface.co/datasets/ControlNet/LAV-DF and click **"Agree and access repository"**.
3. Generate an HF access token: https://huggingface.co/settings/tokens (read-only is fine).
4. In a terminal: `huggingface-cli login` and paste the token (saves to `~/.cache/huggingface/token`).

If you skip step 2 the schema-check cell will fail with `403 Forbidden`. The error message will tell you what to do.

## Why we're using the official splits

LAV-DF's train/dev/test partitions inherit VoxCeleb2's identity-disjoint structure (the source actors in dev/test never appear in train). Using these gives us direct comparability to BA-TFD / BA-TFD+ baselines reported in the original LAV-DF paper — important for the journal submission.

In [ ]:
# --- Imports and paths ---
import os
import sys
import json
import hashlib
import random
import time
from pathlib import Path
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed

# Make sure huggingface_hub is installed in the df env
try:
    from huggingface_hub import snapshot_download, hf_hub_download, HfApi
    from huggingface_hub.utils import HfHubHTTPError
except ImportError:
    print("huggingface_hub not installed. Run in terminal:")
    print("    conda activate df && pip install huggingface_hub")
    raise

import pandas as pd
from tqdm.auto import tqdm

# --- Reproducibility ---
SEED = 42
random.seed(SEED)

# --- Project paths (edit if your layout differs) ---
PROJECT_ROOT = Path(r"C:\Users\Komal-Sch\Desktop\AV-XAI")
DATA_ROOT    = PROJECT_ROOT / "data"
LAVDF_ROOT   = DATA_ROOT / "lav-df"          # where the dataset will live
META_DIR     = LAVDF_ROOT / "_meta"          # metadata.min.json, README.md
SPLITS_DIR   = DATA_ROOT / "splits"
MANIFEST_FP  = DATA_ROOT / "manifest.csv"

for d in (DATA_ROOT, LAVDF_ROOT, META_DIR, SPLITS_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("Project root :", PROJECT_ROOT)
print("Dataset root :", LAVDF_ROOT)
print("Splits dir   :", SPLITS_DIR)
print("Manifest path:", MANIFEST_FP)
print()

# --- Subsample knobs ---
TARGET_TOTAL = 30_000   # bump up later if training time allows
N_CLASSES    = 4        # real, visual_modified, audio_modified, both_modified

# Native LAV-DF split ratios: train 78.7k / dev 31.5k / test 26.1k → 0.58/0.23/0.19
SPLIT_RATIOS = {"train": 0.58, "dev": 0.23, "test": 0.19}
TARGET_PER_SPLIT = {s: int(TARGET_TOTAL * r) for s, r in SPLIT_RATIOS.items()}
TARGET_PER_CELL  = {s: n // N_CLASSES for s, n in TARGET_PER_SPLIT.items()}
print("Target per split  :", TARGET_PER_SPLIT)
print("Target per (split, modify_type) :", TARGET_PER_CELL)

## Phase A — Verify Hugging Face authentication

LAV-DF is a **gated** repo. Without a valid HF token tied to an account that has accepted the terms, every download will 401/403. Fail fast and noisy here so we don't waste minutes downloading and then crash.

In [ ]:
# --- Verify HF auth + dataset access ---
api = HfApi()

try:
    whoami = api.whoami()
    print(f"Logged in as: {whoami['name']} ({whoami.get('fullname', '')})")
except Exception as e:
    print("Not logged in to Hugging Face.")
    print("Fix: run in terminal -> huggingface-cli login")
    print(f"Underlying error: {e}")
    raise

# Probe the dataset itself (this hits the gate)
try:
    info = api.dataset_info(repo_id="ControlNet/LAV-DF")
    print(f"Repo OK     : {info.id}")
    print(f"Last modified: {info.lastModified}")
except HfHubHTTPError as e:
    if "401" in str(e) or "403" in str(e) or "gated" in str(e).lower():
        print("\nHTTP error on the LAV-DF repo — almost certainly the gate:")
        print("  1. Open https://huggingface.co/datasets/ControlNet/LAV-DF in a browser.")
        print("  2. Make sure you are logged in as the SAME account as your CLI token.")
        print("  3. Click 'Agree and access repository'.")
        print("  4. Re-run this cell.")
    raise

## Phase B — Download metadata only (smoke test, ~33 MB)

We pull just `metadata.min.json` and `README.md` first. If the schema is not what we expect, we find out in seconds, not after a 25 GB download.

In [ ]:
# --- Download just the metadata files ---
print("Downloading metadata.min.json and README.md ...")
snapshot_download(
    repo_id="ControlNet/LAV-DF",
    repo_type="dataset",
    local_dir=str(META_DIR),
    local_dir_use_symlinks=False,
    allow_patterns=["metadata.min.json", "README.md"],
    resume_download=True,
)
print("Done.")

# Sanity check that the file exists and is non-empty
meta_min_fp = META_DIR / "metadata.min.json"
assert meta_min_fp.exists(), f"metadata.min.json missing at {meta_min_fp}"
print(f"metadata.min.json size: {meta_min_fp.stat().st_size / 1e6:.2f} MB")

## Phase C — Inspect the metadata schema

Print the field set of a few entries (one per class) and assert the fields we depend on are present. If the schema has drifted from what we expect, this cell raises with a precise message.

In [ ]:
# --- Load and inspect metadata ---
with open(meta_min_fp, "r", encoding="utf-8") as f:
    meta = json.load(f)

print(f"Total entries: {len(meta):,}")
print(f"Top-level type: {type(meta).__name__}")
print()

# Show ALL fields seen across the first 100 entries (catches optional fields)
all_fields = set()
for entry in meta[:100]:
    all_fields.update(entry.keys())
print(f"Fields observed in first 100 entries: {sorted(all_fields)}")
print()

# --- Print one example per modify_type ---
seen_types = {}
for entry in meta:
    mt = entry.get("modify_type", entry.get("type", None))
    if mt not in seen_types:
        seen_types[mt] = entry
    if len(seen_types) >= 4:
        break

print("--- One example per modify_type ---")
for mt, entry in seen_types.items():
    print(f"\nmodify_type = {mt!r}")
    print(json.dumps(entry, indent=2)[:600])

# --- Assert the fields we need ---
REQUIRED_FIELDS = ("file", "modify_type", "split", "original")
sample = meta[0]
missing = [f for f in REQUIRED_FIELDS if f not in sample]
if missing:
    print(f"\n!! Missing required fields in metadata: {missing}")
    print(f"   Got fields: {sorted(sample.keys())}")
    print("   You probably need to rename our column expectations. Inspect 'meta[0]' above.")
    raise AssertionError(f"Schema mismatch: missing {missing}")
print("\nSchema check passed.")

## Phase D — Class distribution, actor counts, and actor-disjointness check

Three things to verify:
1. **Are the four classes all well-populated?** If one class has <7,500 in train, we'll have to cap.
2. **How many unique actors per split?**
3. **Are the official splits actor-disjoint?** (Should be, by VoxCeleb2 inheritance. Verify, don't assume.)

In [ ]:
# --- Actor ID extraction (VoxCeleb2 layout: idXXXXX/yt_video_id/clip.mp4) ---
def extract_actor_id(entry):
    """
    Return the VoxCeleb2 speaker id (e.g. 'id00012') for a LAV-DF entry.
    Real videos: the 'original' field is the source path in VoxCeleb2.
    Fake videos: the 'original' field is the source REAL video, also in VoxCeleb2.
    So in both cases, parsing 'original' gives us the speaker.
    """
    orig = entry.get("original")
    if orig is None:
        return None
    parts = Path(orig).parts
    # Find the first part that looks like an id (starts with 'id' and is alphanumeric)
    for p in parts:
        if p.startswith("id") and p[2:].isdigit():
            return p
    # Fallback: first path part
    return parts[0] if parts else None

# --- Tag every entry with actor_id, group by (split, modify_type) ---
buckets = defaultdict(list)              # (split, modify_type) -> list of entries
actors_per_split = defaultdict(set)      # split -> set of actor_ids

for entry in tqdm(meta, desc="Indexing metadata"):
    split = entry["split"]
    mt    = entry["modify_type"]
    actor = extract_actor_id(entry)
    entry["_actor_id"] = actor          # cache for later
    buckets[(split, mt)].append(entry)
    if actor is not None:
        actors_per_split[split].add(actor)

# --- Print distribution table ---
print("Class distribution per split:")
print(f"{'split':<10} {'real':>10} {'visual_mod':>12} {'audio_mod':>12} {'both_mod':>10} {'TOTAL':>10}")
print("-" * 66)
totals = Counter()
for split in ("train", "dev", "test"):
    row = [split]
    split_total = 0
    for mt in ("real", "visual_modified", "audio_modified", "both_modified"):
        n = len(buckets[(split, mt)])
        row.append(n)
        split_total += n
        totals[mt] += n
    row.append(split_total)
    totals["TOTAL"] += split_total
    print(f"{row[0]:<10} {row[1]:>10,} {row[2]:>12,} {row[3]:>12,} {row[4]:>10,} {row[5]:>10,}")
print("-" * 66)
print(f"{'TOTAL':<10} {totals['real']:>10,} {totals['visual_modified']:>12,} "
      f"{totals['audio_modified']:>12,} {totals['both_modified']:>10,} {totals['TOTAL']:>10,}")

# --- Unique actors per split ---
print()
for s in ("train", "dev", "test"):
    print(f"Unique actors in {s}: {len(actors_per_split[s]):,}")

# --- Actor-disjointness check ---
print("\n--- Actor leakage check (should be all 0) ---")
overlaps = {
    "train ∩ dev" : actors_per_split["train"] & actors_per_split["dev"],
    "train ∩ test": actors_per_split["train"] & actors_per_split["test"],
    "dev   ∩ test": actors_per_split["dev"]   & actors_per_split["test"],
}
for name, overlap in overlaps.items():
    flag = "OK" if len(overlap) == 0 else "!! LEAKAGE"
    print(f"  {name}: {len(overlap):>4} shared actors  [{flag}]")
    if overlap and len(overlap) <= 5:
        print(f"      examples: {list(overlap)[:5]}")
assert all(len(v) == 0 for v in overlaps.values()), "Official splits are NOT actor-disjoint. Investigate before proceeding."
print("\nOfficial splits are actor-disjoint.")

## Phase E — Subsample for balanced 30K

Within each official split, take up to `TARGET_PER_CELL[split]` entries from each `modify_type`. If a cell has fewer entries than the target, we use all of them and print a warning (no upsampling — that would bias training).

Random shuffle is seeded; reproducible.

In [ ]:
# --- Balanced subsample ---
rng = random.Random(SEED)
sampled = []
warnings_list = []

for split in ("train", "dev", "test"):
    target = TARGET_PER_CELL[split]
    for mt in ("real", "visual_modified", "audio_modified", "both_modified"):
        pool = buckets[(split, mt)]
        if len(pool) >= target:
            picked = rng.sample(pool, target)
        else:
            picked = list(pool)
            warnings_list.append(
                f"  ({split}, {mt}): only {len(pool):,} available; wanted {target:,}. Using all."
            )
        sampled.extend(picked)

if warnings_list:
    print("Warnings (class under-target):")
    for w in warnings_list:
        print(w)
else:
    print("All target cells filled.")

print(f"\nTotal sampled entries: {len(sampled):,}")

# Re-check sampled distribution
sc = Counter((e["split"], e["modify_type"]) for e in sampled)
print("\nSampled distribution:")
for split in ("train", "dev", "test"):
    for mt in ("real", "visual_modified", "audio_modified", "both_modified"):
        print(f"  ({split:<5}, {mt:<16}): {sc[(split, mt)]:,}")

## Phase F — Build `manifest.csv` and `splits/{train,val,test}.csv`

Schema for the manifest:
| col | meaning |
|---|---|
| `file` | relative path under `data/lav-df/`, e.g. `train/000001.mp4` |
| `split` | `train` / `val` / `test` (we rename LAV-DF's `dev` → `val` for our pipeline consistency) |
| `modify_type` | one of the 4 string labels |
| `label_4class` | int 0–3 (real=0, visual=1, audio=2, both=3) |
| `label_binary` | int 0/1 (real=0, any-fake=1) |
| `actor_id` | VoxCeleb2 speaker id |
| `fake_periods` | JSON-serialized list of `[start_s, end_s]` for temporal localization (empty for real) |
| `n_fakes` | int count of fake segments |
| `original` | original VoxCeleb2 source path |

In [ ]:
# --- Build manifest rows ---
LABEL_4 = {"real": 0, "visual_modified": 1, "audio_modified": 2, "both_modified": 3}
SPLIT_RENAME = {"train": "train", "dev": "val", "test": "test"}   # LAV-DF dev -> our val

rows = []
for entry in sampled:
    mt = entry["modify_type"]
    rows.append({
        "file"         : entry["file"],
        "split"        : SPLIT_RENAME[entry["split"]],
        "modify_type"  : mt,
        "label_4class" : LABEL_4[mt],
        "label_binary" : 0 if mt == "real" else 1,
        "actor_id"     : entry.get("_actor_id"),
        "fake_periods" : json.dumps(entry.get("fake_periods", [])),
        "n_fakes"      : entry.get("n_fakes", len(entry.get("fake_periods", []))),
        "original"     : entry.get("original"),
    })

df = pd.DataFrame(rows).sort_values(["split", "modify_type", "file"]).reset_index(drop=True)
df.to_csv(MANIFEST_FP, index=False)
print(f"Wrote manifest: {MANIFEST_FP} ({len(df):,} rows)")

# Per-split CSVs (just the file + label columns for fast loader use)
for s in ("train", "val", "test"):
    sub = df[df["split"] == s].reset_index(drop=True)
    out_fp = SPLITS_DIR / f"{s}.csv"
    sub.to_csv(out_fp, index=False)
    print(f"  {out_fp.name}: {len(sub):,} rows")

## Phase G — Selectively download only the videos in the manifest

This is the slow phase. We use a thread pool because most of the bottleneck is HTTPS round-trips. Adjust `MAX_WORKERS` based on the lab network — start with 8, bump to 16 if it's stable.

`hf_hub_download` is resumable and caches locally, so re-running this cell after an interruption just picks up where it left off.

**Disk math at this point:**  ~30K mp4s × ~190 KB avg = ~5.7 GB raw.

In [ ]:
# --- Selective download with thread pool ---
MAX_WORKERS = 8

files_to_get = df["file"].tolist()
print(f"Files to download: {len(files_to_get):,}")
print(f"Workers: {MAX_WORKERS}")

# Skip files already present
def already_downloaded(rel):
    fp = LAVDF_ROOT / rel
    return fp.exists() and fp.stat().st_size > 0

missing = [f for f in files_to_get if not already_downloaded(f)]
print(f"Already on disk: {len(files_to_get) - len(missing):,}")
print(f"Need to fetch  : {len(missing):,}")

if not missing:
    print("All files already present; skipping download.")
else:
    def download_one(rel_path):
        try:
            hf_hub_download(
                repo_id="ControlNet/LAV-DF",
                repo_type="dataset",
                filename=rel_path,
                local_dir=str(LAVDF_ROOT),
                local_dir_use_symlinks=False,
                resume_download=True,
            )
            return (rel_path, None)
        except Exception as e:
            return (rel_path, str(e))

    failures = []
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        futures = [pool.submit(download_one, f) for f in missing]
        for fut in tqdm(as_completed(futures), total=len(futures), desc="Downloading"):
            rel, err = fut.result()
            if err:
                failures.append((rel, err))

    elapsed = time.time() - t0
    print(f"\nDone in {elapsed/60:.1f} min")
    print(f"Failures: {len(failures)}")
    if failures[:5]:
        print("First few failures:")
        for rel, err in failures[:5]:
            print(f"  {rel}: {err[:160]}")
    # Save failures for retry
    if failures:
        fail_fp = LAVDF_ROOT / "_download_failures.txt"
        with open(fail_fp, "w") as f:
            for rel, err in failures:
                f.write(f"{rel}\t{err}\n")
        print(f"Failures logged to {fail_fp}. Re-run this cell to retry.")

## Phase H — Final verification

Three sanity checks:
1. Every file in the manifest exists on disk and is non-empty.
2. No actor leaks across splits (re-check after subsampling — should be identical to Phase D, but verify).
3. Class balance is within 1% of target per cell.

In [ ]:
# --- Verification ---
print("--- 1. File existence + size check ---")
missing_on_disk = []
zero_bytes = []
for rel in tqdm(df["file"], desc="Checking files"):
    fp = LAVDF_ROOT / rel
    if not fp.exists():
        missing_on_disk.append(rel)
    elif fp.stat().st_size == 0:
        zero_bytes.append(rel)

print(f"Missing files : {len(missing_on_disk)}")
print(f"Zero-byte     : {len(zero_bytes)}")
if missing_on_disk[:3]:
    print(f"  e.g. {missing_on_disk[:3]}")
assert not missing_on_disk and not zero_bytes, "File integrity failed; re-run Phase G."

print("\n--- 2. Actor leakage check (post-subsample) ---")
actors = {s: set(df[df["split"] == s]["actor_id"]) for s in ("train", "val", "test")}
for a, b in [("train","val"), ("train","test"), ("val","test")]:
    overlap = actors[a] & actors[b]
    flag = "OK" if not overlap else "!! LEAKAGE"
    print(f"  {a} ∩ {b}: {len(overlap):>3} shared actors  [{flag}]")
    assert not overlap, f"Actor leakage between {a} and {b}"

print("\n--- 3. Class balance check ---")
for s in ("train", "val", "test"):
    counts = df[df["split"] == s]["modify_type"].value_counts()
    print(f"  {s}: {dict(counts)}")
    if len(counts) == 4 and counts.min() > 0:
        spread = (counts.max() - counts.min()) / counts.max()
        print(f"    balance spread (max-min)/max = {spread:.1%}")

print("\nAll verification checks passed. Ready for notebook 02 (preprocessing).")

## Done.

**What's on disk now:**
```
C:\Users\Komal-Sch\Desktop\AV-XAI\data\
├── lav-df/
│   ├── _meta/
│   │   ├── metadata.min.json
│   │   └── README.md
│   ├── train/*.mp4   (subset, ~17K files)
│   ├── dev/*.mp4     (subset, ~7K files)
│   └── test/*.mp4    (subset, ~6K files)
├── splits/
│   ├── train.csv
│   ├── val.csv
│   └── test.csv
└── manifest.csv
```

**Next step (notebook 02):**
- Update `02_preprocessing.ipynb` paths to point at `data/lav-df/` instead of the old Celeb-DF paths
- MTCNN face detection logic stays identical
- Audio extraction (librosa / `av.open()`) will *actually work this time* since every LAV-DF clip has audio by construction
- Output preprocessed faces → `data/preprocessed/faces/`, audio → `data/preprocessed/audio/`

**Update `project_log.md`** with the date, total downloaded, and any cells that emitted warnings.

**Git checkpoint:**
```
git add notebooks/01_setup_and_download_lavdf.ipynb data/manifest.csv data/splits/*.csv
git commit -m "nb01: LAV-DF download + balanced 30K manifest with official actor-disjoint splits"
git push
```

⚠️ Do **not** commit the actual mp4 files — they're under CC BY-NC, and the repo is already large enough. Add `data/lav-df/**/*.mp4` to `.gitignore` if it isn't there.